In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import math

In [ ]:
input_dim = 32

#teacher
torch.manual_seed(3948284)
teacher = nn.Sequential(nn.Linear(input_dim, 256), nn.GELU(),
                        nn.Linear(256, 256), nn.GELU(),
                        nn.Linear(256, 1))
#Normalization
with torch.no_grad():
    inp = torch.randn(65536, input_dim)
    out = teacher(inp)
out_mean = out.mean().item()
out_scale = 1.0/out.std().item()

@torch.no_grad()
def norm_target(x):
    return (teacher(x) - out_mean)*out_scale

In [ ]:
def cosine_lr_lambda(total_steps: int, warmup_frac: float = 0.08,
                     start_factor: float = 0.01, end_factor: float = 0.1):
    """LR multiplier: linear warmup eta/100 -> eta over ``warmup_frac`` of the steps,
    then cosine annealing eta -> eta/10. The per-cell cosine schedule used throughout."""
    warmup_steps = max(1, int(warmup_frac * total_steps))

    def fn(step: int) -> float:
        if step < warmup_steps:
            return start_factor + (1.0 - start_factor) * step / warmup_steps
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return end_factor + (1.0 - end_factor) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return fn

In [ ]:
def make_std():
    torch.manual_seed(456394)
    student = nn.Sequential(nn.Linear(input_dim, 64), nn.GELU(),
                            nn.Linear(64, 64), nn.GELU(),
                            nn.Linear(64, 1))
    return student

In [ ]:
def train_student(lr, n_train_steps):
    mlp = make_std()
    loss_function = nn.MSELoss()
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=0.0)
    n_batch = 256

    for i in range(n_train_steps):
        x = torch.randn(n_batch, input_dim)
        target = norm_target(x)

        optimizer.zero_grad()
        loss = loss_function(mlp(x), target)
        loss.backward()
        optimizer.step()
    
    #validation set
    val_size = 16384
    gv = torch.Generator().manual_seed(2876793)
    x_val = torch.randn(val_size, input_dim, generator=gv)
    target_val = norm_target(x_val)
    val_loss = loss_function(mlp(x_val), target_val)

    return val_loss.item()

In [ ]:
def train_student_sched(lr, n_train_steps):
    mlp = make_std()
    loss_function = nn.MSELoss()
    optimizer = torch.optim.AdamW(mlp.parameters(), lr, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, cosine_lr_lambda(n_train_steps))
    n_batch = 256

    for i in range(n_train_steps):
            x = torch.randn(n_batch, input_dim)
            target = norm_target(x)
    
            optimizer.zero_grad()
            loss = loss_function(mlp(x), target)
            loss.backward()
            optimizer.step()
            scheduler.step()
        
    #validation set
    val_size = 16384
    gv = torch.Generator().manual_seed(2876793)
    x_val = torch.randn(val_size, input_dim, generator=gv)
    target_val = norm_target(x_val)
    val_loss = loss_function(mlp(x_val), target_val)

    return val_loss.item()


In [ ]:
#evaluation of val_loss for different lrs
learn_rates = np.geomspace(3e-4, 1e-1, 9)
losses = []
for lr in learn_rates:
    val_loss = train_student(lr, 2048)
    losses.append(val_loss)

losses_sched=[]
for lr in learn_rates:
    val_loss = train_student_sched(lr, 2048)
    losses_sched.append(val_loss)


In [ ]:
def plot_lr_tuning(lrs, losses, losses_sched):
    lrs = np.asarray(lrs, dtype="float")
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(lrs, losses, "o-", color="steelblue", ms=7, label="constant lr")
    ax.plot(lrs, losses_sched, "o-", color="indianred", ms=7, label="cosine schedule")
    ax.set(xscale="log", xlabel=r"learning rate $\eta$", ylabel="validation loss")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

plot_lr_tuning(learn_rates, losses, losses_sched)